## Configuration & Usage


**How to use it:**
- Modify **only Cell 3** (User Parameters)
- Set `MOCK_MODE = True` for fast testing (no API needed)
- Set `MOCK_MODE = False` for real ERA5 downloads (requires CDS API key)
- Run "Run All" → pipeline executes automatically

**CDS API setup** (for real mode only):
1. Register free account: https://cds.climate.copernicus.eu
2. Get API key from Profile 
3. Create `~/.cdsapirc`:
```
url: https://cds.climate.copernicus.eu/api/v2
key: YOUR_UID:YOUR_API_KEY
```

In [8]:
%matplotlib inline

# ============================================================================
# USER PARAMETERS - EDIT THIS CELL FOR NORMAL USAGE
# ============================================================================

# 1. MOCK MODE: True for testing, False for real CDS API downloads
MOCK_MODE = False

# 2. ERA5 CONFIGURATION
# For 2m temperature (single-level variable) we request the CDS single-level product
ERA5_CONFIG = {
    'variable': '2m_temperature',            # ERA5 variable name for 2m temperature
    'pressure_levels': [],                   # Empty for single-level variables
    'single_level': True,                    # Use CDS single-level product
    'times': ['00:00', '06:00', '12:00', '18:00'],  # Times (UTC)
    'format': 'netcdf',                      # Format: 'netcdf' or 'grib'
    'grid': None,                            # None = global, or [N, W, S, E] for subset
}

# 3. DATE RANGE or Single Date
# Leave all as None to use hardcoded defaults (2024-12-01 to 2024-12-05)
from datetime import datetime, timedelta
single_date = None    # Set to process one specific date
start_date =  datetime(2026, 1, 6)   # Set to specify custom range start
end_date =   datetime(2026, 1, 19)   # Set to specify custom range end
# Default demo range (edit if you want a different built-in default)
DEFAULT_START_DATE = datetime(2024, 12, 1)
DEFAULT_END_DATE = datetime(2024, 12, 5)
# Examples:
# - All None (default): Uses hardcoded range 2024-12-01 to 2024-12-05
# - single_date = datetime(2024, 12, 1): Downloads only that date
# - start_date = datetime(2024, 12, 10), end_date = datetime(2024, 12, 15): Custom range


## Function Definitions

The cells below define all functions used by the pipeline. Do not modify these cells.

In [9]:
# ============================================================================
# SETUP & INITIALIZATION 
# ============================================================================

from pathlib import Path
from typing import List, Optional
import logging
import shutil

# Project directories
DATA_DIR_BASE = Path.cwd().parent / 'data_access' / 'data'
DATA_DIR_MOCK = DATA_DIR_BASE / 'mock'
DATA_DIR_REAL = DATA_DIR_BASE / 'real'

# Select data directory based on MOCK_MODE
DATA_DIR = DATA_DIR_MOCK if MOCK_MODE else DATA_DIR_REAL

ARCHIVE_DIR_BASE = Path.cwd().parent / 'data_access' / 'archive'
ARCHIVE_DIR_MOCK = ARCHIVE_DIR_BASE / 'mock'
ARCHIVE_DIR_REAL = ARCHIVE_DIR_BASE / 'real'
ARCHIVE_DIR = ARCHIVE_DIR_MOCK if MOCK_MODE else ARCHIVE_DIR_REAL

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('era5_download')

# Ensure directories exist
for d in [DATA_DIR, DATA_DIR_MOCK, DATA_DIR_REAL, ARCHIVE_DIR, ARCHIVE_DIR_MOCK, ARCHIVE_DIR_REAL]:
    d.mkdir(parents=True, exist_ok=True)

# Try to import xarray for validation
try:
    import xarray as xr
    _HAS_XARRAY = True
except ImportError:
    _HAS_XARRAY = False
    logger.warning("xarray not available - file validation will be skipped")

In [10]:
# Download routine (uses cdsapi.Client when MOCK_MODE=False)
import json

try:
    import cdsapi
    _HAS_CDSAPI = True
except Exception:
    _HAS_CDSAPI = False


def _is_valid_era5_file(file_path: Path) -> bool:
    """
    Check if ERA5 file is valid by attempting to open it.
    
    Validates that:
    - File can be opened as a NetCDF/GRIB dataset
    - Has data variables
    - Has expected coordinate dimensions (latitude/longitude)
    
    Args:
        file_path: Path to ERA5 file to validate
    
    Returns:
        True if file is valid, False otherwise
    """
    if not _HAS_XARRAY:
        # If can't validate, conservatively assume valid (avoid blocking downloads)
        logger.warning(f"Cannot validate {file_path.name}: xarray not available")
        return True
    
    try:
        with xr.open_dataset(file_path) as ds:
            # Check has data variables
            if len(ds.data_vars) == 0:
                logger.warning(f"File has no data variables: {file_path.name}")
                return False
            
            # Check has coordinate dimensions (latitude/longitude or lat/lon)
            has_coords = False
            for var in ds.data_vars:
                var_coords = ds[var].coords
                if ('latitude' in var_coords and 'longitude' in var_coords) or \
                   ('lat' in var_coords and 'lon' in var_coords):
                    has_coords = True
                    break
            
            if not has_coords:
                logger.warning(f"File missing lat/lon coordinates: {file_path.name}")
                return False
            
            return True
            
    except Exception as e:
        logger.warning(f"File validation failed for {file_path.name}: {e}")
        return False


def download_era5_daily(
    date: datetime,
    output_dir: Path = None,
    mock: bool = None,
) -> Optional[Path]:
    """
    Download a single day of ERA5 data or create a mock file.
    
    Validates existing files before returning to ensure interrupted downloads
    are detected and re-downloaded.
    
    Args:
        date: Date to download
        output_dir: Directory to save file (defaults to DATA_DIR based on MOCK_MODE)
        mock: Override global MOCK_MODE (None = use global setting)
    
    Returns:
        Path to downloaded or created file, or None if failed
    
    Note: This function uses configuration from ERA5_CONFIG.
    """
    if output_dir is None:
        output_dir = DATA_DIR
    if mock is None:
        mock = MOCK_MODE
    
    # Extract configuration from ERA5_CONFIG
    variable = ERA5_CONFIG['variable']
    pressure_levels = ERA5_CONFIG.get('pressure_levels', [])
    times = ERA5_CONFIG['times']
    output_format = ERA5_CONFIG['format']
    grid = ERA5_CONFIG['grid']
    
    date_str = date.strftime('%Y%m%d')
    ext = 'nc' if output_format == 'netcdf' else 'grib'
    output_path = output_dir / f"era5_{variable}_{date_str}.{ext}"

    # Check if file exists AND is valid
    if output_path.exists():
        if _is_valid_era5_file(output_path):
            logger.info(f"File already exists and is valid: {output_path}")
            return output_path
        else:
            logger.warning(f"Existing file is invalid or incomplete, will re-download: {output_path}")
            output_path.unlink()  # Delete invalid file

    output_dir.mkdir(parents=True, exist_ok=True)

    if mock:
        # Create an empty mock file
        output_path.touch()
        logger.info(f"[MOCK] Created mock file: {output_path}")
        return output_path

    # Real download path
    if not _HAS_CDSAPI:
        logger.error('cdsapi is not installed in this environment. Install with `pip install cdsapi`')
        return None

    client = cdsapi.Client()

    # Choose CDS product depending on whether this is a single-level variable
    if ERA5_CONFIG.get('single_level', False):
        product_name = 'reanalysis-era5-single-levels'
    else:
        product_name = 'reanalysis-era5-pressure-levels'

    # Build request dictionary; omit pressure_level for single-level products
    request = {
        'product_type': 'reanalysis',
        'format': output_format,
        'variable': variable,
        'date': date.strftime('%Y-%m-%d'),
        'time': times,
    }
    if not ERA5_CONFIG.get('single_level', False):
        request['pressure_level'] = pressure_levels
    if grid is not None:
        request['grid'] = grid

    try:
        logger.info(f"Downloading {date.strftime('%Y-%m-%d')}: {variable} product={product_name}")
        client.retrieve(product_name, request, str(output_path))

        # Validate downloaded file before returning
        if not _is_valid_era5_file(output_path):
            logger.error(f"Downloaded file failed validation: {output_path}")
            output_path.unlink()  # Clean up invalid download
            return None

        logger.info(f"Downloaded to {output_path}")
        return output_path
    except Exception as e:
        logger.error(f"Download failed: {e}")
        # Clean up partial file if it exists
        if output_path.exists():
            output_path.unlink()
        return None


## Archiving
Archive downloaded ERA5 files by date (YYYY/MM/ structure).

In [11]:
def archive_file(downloaded_file: Path, mock: bool = None) -> Optional[Path]:
    """
    Move downloaded file to archive directory organized by YYYY/MM/.
    
    Args:
        downloaded_file: Path to downloaded file
        mock: Use mock or real archive directory (None = use MOCK_MODE)
    
    Returns:
        Path to archived file, or None if failed
    """
    if mock is None:
        mock = MOCK_MODE
    
    archive_dir = ARCHIVE_DIR_MOCK if mock else ARCHIVE_DIR_REAL
    
    if not downloaded_file.exists():
        logger.error(f"Downloaded file not found: {downloaded_file}")
        return None
    
    try:
        # Extract date from filename (e.g., era5_relative_humidity_20241201.nc)
        filename = downloaded_file.name
        # Find YYYYMMDD pattern in filename
        import re
        date_match = re.search(r'(\d{8})', filename)
        if not date_match:
            logger.error(f"Could not extract date from filename: {filename}")
            return None
        
        date_str = date_match.group(1)
        yyyy = date_str[:4]
        mm = date_str[4:6]
        
        # Create archive subdirectory
        archive_subdir = archive_dir / yyyy / mm
        archive_subdir.mkdir(parents=True, exist_ok=True)
        
        # Move file to archive
        archived_path = archive_subdir / filename
        shutil.move(str(downloaded_file), str(archived_path))
        
        logger.info(f"Archived to: {archived_path}")
        return archived_path
        
    except Exception as e:
        logger.error(f"Failed to archive {downloaded_file}: {e}")
        return None

## Pipeline orchestration (functions)
These functions let you download a single date, or a date range. Pass `mock=True` to skip actual downloads.

In [12]:
def check_day_completeness(date: datetime, archive_dir: Path = None) -> dict:
    """
    Check if a day's file exists in the archive.
    
    Args:
        date: Date to check
        archive_dir: Archive directory to search in (auto-detects mock/real if None)
    
    Returns:
        Dictionary with keys:
        - 'complete': bool (True if file present in archive)
        - 'archive_path': Path where file should be/was found
    """
    if archive_dir is None:
        archive_dir = ARCHIVE_DIR_REAL  # Default to real archive for checking
    
    variable = ERA5_CONFIG['variable']
    yyyy = date.strftime('%Y')
    mm = date.strftime('%m')
    
    archive_date_folder = archive_dir / yyyy / mm
    date_str = date.strftime('%Y%m%d')
    ext = 'nc' if ERA5_CONFIG['format'] == 'netcdf' else 'grib'
    
    # Look for archived file with this date
    if not archive_date_folder.exists():
        return {
            'complete': False,
            'archive_path': archive_date_folder
        }
    
    # Check if the file exists in archive
    archived_file = archive_date_folder / f"era5_{variable}_{date_str}.{ext}"
    
    return {
        'complete': archived_file.exists(),
        'archive_path': archive_date_folder
    }


def should_skip_date(date: datetime, mock: bool = None) -> bool:
    """
    Determine if a date should be skipped (already downloaded).
    
    Only skips if file exists in archive.
    
    Args:
        date: Date to check
        mock: Check mock archive if True, real if False (None = use MOCK_MODE)
    
    Returns:
        True if date should be skipped, False if it should be downloaded
    """
    if mock is None:
        mock = MOCK_MODE
    
    archive_dir = ARCHIVE_DIR_MOCK if mock else ARCHIVE_DIR_REAL
    completeness = check_day_completeness(date, archive_dir)
    
    if completeness['complete']:
        logger.info(f"Skipping {date.strftime('%Y-%m-%d')}: already in archive")
        return True
    
    logger.info(f"Downloading {date.strftime('%Y-%m-%d')}: not yet in archive")
    return False


def process_single_date(date: datetime, mock: bool = None, skip_if_complete: bool = True) -> bool:
    """
    Download and archive a single date.
    
    Args:
        date: Date to process
        mock: Override global MOCK_MODE (None = use global setting)
        skip_if_complete: Skip if day is already in archive (default: True)
    
    Returns:
        True if successful or skipped as complete, False otherwise
    """
    if mock is None:
        mock = MOCK_MODE
    
    # Skip if day is already in archive
    if skip_if_complete and should_skip_date(date, mock=mock):
        return True  # Treat as success since it's already done
    
    downloaded = download_era5_daily(date=date, mock=mock)
    if downloaded is None:
        return False
    
    archived = archive_file(downloaded, mock=mock)
    return archived is not None


def process_date_range(start_date: datetime, end_date: datetime, mock: bool = None):
    """
    Download a range of dates with intelligent skipping of complete dates.
    
    Args:
        start_date: Start date (inclusive)
        end_date: End date (inclusive)
        mock: Override global MOCK_MODE (None = use global setting)
    
    Returns:
        Dictionary with counts:
        - 'downloaded': dates that were actually downloaded
        - 'skipped': dates that were already in archive
        - 'failed': dates that failed to download
    """
    if mock is None:
        mock = MOCK_MODE
    
    current = start_date
    results = {'downloaded': 0, 'skipped': 0, 'failed': 0}
    
    while current <= end_date:
        # Check if day should be skipped (already in archive)
        if should_skip_date(current, mock=mock):
            results['skipped'] += 1
        else:
            # Download the date
            ok = process_single_date(current, mock=mock, skip_if_complete=False)  # Already checked
            if ok:
                results['downloaded'] += 1
            else:
                results['failed'] += 1
        
        current += timedelta(days=1)
    
    return results

In [13]:
def find_oldest_missing_file(
    start_date: datetime = DEFAULT_START_DATE,
    end_date: datetime = DEFAULT_END_DATE,
    mock: bool = None,
    variable: str = ERA5_CONFIG['variable']
) -> Optional[datetime]:
    """
    Scan the date range and return the first missing date.
    A date is considered "missing" if not yet archived.
    
    Returns None if all dates are in archive.
    """
    if mock is None:
        mock = MOCK_MODE
    
    current = start_date
    ext = 'nc' if ERA5_CONFIG['format'] == 'netcdf' else 'grib'
    archive_dir = ARCHIVE_DIR_MOCK if mock else ARCHIVE_DIR_REAL
    
    while current <= end_date:
        # Check archive for completeness
        completeness = check_day_completeness(current, archive_dir)
        
        # If missing, this is the oldest missing date
        if not completeness['complete']:
            logger.info(f"Found missing date: {current.strftime('%Y-%m-%d')}")
            return current
        
        current += timedelta(days=1)
    
    logger.info("No missing files found")
    return None

## Pipeline Control Flow

The entire ERA5 download workflow is encapsulated in a single function, run_era5_pipeline(), which serves as the main entry point for downloading and archiving daily data.

In [14]:
def run_era5_pipeline(single_date=None, start_date=None, end_date=None, mock=None):
    """
    Main pipeline function: download and archive ERA5 data.
    
    This function encapsulates the entire workflow in a single call.
    It downloads ERA5 data for the specified date range (or hardcoded defaults)
    and organizes files in archive directories.
    
    AUTOMATIC FEATURES (no user action required):
    ✓ Skips dates that are already in archive
    ✓ Validates downloaded files before archiving
    
    Args:
        single_date (datetime, optional): Download only this specific date.
            Takes precedence over start_date/end_date if provided.
        start_date (datetime, optional): Start of date range. 
            Default: Uses DEFAULT_START_DATE (2024-12-01)
        end_date (datetime, optional): End of date range. 
            Default: Uses DEFAULT_END_DATE (2024-12-05)
        mock (bool, optional): Use mock mode (True) or real CDS API (False). 
            Default: Uses global MOCK_MODE from Cell 3
    
    Returns:
        dict: Results with keys 'downloaded', 'skipped', 'failed'
    """
    # Handle single_date parameter (takes precedence)
    if single_date is not None:
        start_date = single_date
        end_date = single_date
    
    # Use hardcoded defaults from setup cell if not provided
    if start_date is None:
        start_date = DEFAULT_START_DATE
    if end_date is None:
        end_date = DEFAULT_END_DATE
    if mock is None:
        mock = MOCK_MODE
    
    # Log pipeline configuration
    logger.info("\n" + "=" * 70)
    logger.info("ERA5 DOWNLOAD PIPELINE")
    logger.info("=" * 70)
    logger.info(f"Start date: {start_date.date()}")
    logger.info(f"End date: {end_date.date()}")
    logger.info(f"Mode: {'MOCK' if mock else 'REAL'}")
    logger.info(f"Variable: {ERA5_CONFIG['variable']}")
    
    # This automatically handles:
    # 1. Checking each date for existence in archive
    # 2. Skipping dates already archived
    # 3. Downloading missing dates
    results = process_date_range(start_date, end_date, mock=mock)
    
    # Log results
    logger.info("\n" + "=" * 70)
    logger.info("PIPELINE RESULTS")
    logger.info("=" * 70)
    logger.info(f"✓ Downloaded: {results['downloaded']} days")
    logger.info(f"⊘ Skipped (already in archive): {results['skipped']} days")
    logger.info(f"✗ Failed: {results['failed']} days")
    logger.info("=" * 70 + "\n")
    
    return results


# ============================================================================
# EXECUTE THE PIPELINE
# ============================================================================
# Run the pipeline with parameters from Cell 3 or with hardcoded defaults

if single_date is not None:
    # Single date mode
    logger.info(f"Running pipeline for single date: {single_date.date()}")
    results = run_era5_pipeline(single_date=single_date)
elif start_date is not None and end_date is not None:
    # Range mode
    logger.info(f"Running pipeline with specified date range")
    results = run_era5_pipeline(start_date=start_date, end_date=end_date)
else:
    # Default mode: use hardcoded defaults
    logger.info(f"Running pipeline with hardcoded defaults (2024-12-01 to 2024-12-05)")
    results = run_era5_pipeline()

2026-01-21 18:45:42,562 - INFO - Running pipeline with specified date range
2026-01-21 18:45:42,566 - INFO - 
2026-01-21 18:45:42,575 - INFO - ERA5 DOWNLOAD PIPELINE
2026-01-21 18:45:42,576 - INFO - ======================================================================
2026-01-21 18:45:42,576 - INFO - Start date: 2026-01-06
2026-01-21 18:45:42,577 - INFO - End date: 2026-01-19
2026-01-21 18:45:42,577 - INFO - Mode: REAL
2026-01-21 18:45:42,578 - INFO - Variable: 2m_temperature
2026-01-21 18:45:42,578 - INFO - Downloading 2026-01-06: not yet in archive
2026-01-21 18:45:44,009 - INFO - Downloading 2026-01-06: 2m_temperature product=reanalysis-era5-single-levels
2026-01-21 18:45:45,030 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended 